# Master's Thesis — QSAR Pipeline for Drug Repositioning in Cystic Fibrosis

**Author:** Vicente Valero Just

**MSc in Bioinformatics and Biostatistics (UOC, 2025–2026)**

---

## Structure

1. Setup and libraries
2. Reusable functions
3. Dataset 1 — ChEMBL
4. Dataset 2 — Papyrus
5. Dataset 3 — BindingDB
6. Merging the 3 databases
7. Model 1 — ChEMBL-only baseline
8. Model 2 — QSAR with the merged dataset (+ model comparison)
9. Candidates and validation controls
10. Applicability domain (Tanimoto)
11. pIC50 prediction and integration with docking
12. Model validation with the controls
13. Final ranking
14. Visualizations
15. Export results
16. Comparative figure of the four PLIP profiles

---
## 1. Setup and libraries

If the dependencies are already installed, the first cell does nothing.

In [ ]:
# Install dependencies (uncomment the first time)
# !pip install chembl-webresource-client papyrus-scripts rdkit scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
# Data handling
import pandas as pd
import numpy as np
import time
import os

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

# Computational chemistry (RDKit)
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski, AllChem, rdMolDescriptors

# Machine Learning (scikit-learn)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

# ChEMBL client
from chembl_webresource_client.new_client import new_client

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Fingerprint parameters (the same throughout the notebook)
FP_SIZE = 1024
FP_RADIUS = 2

# Input/output folders
DATA_DIR = "data"        # place bindingdb_cftr.tsv here (see README)
RESULTS_DIR = "results"  # generated CSVs and figures
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Libraries loaded. SEED =", SEED)

---
## 2. Functions

We define them once at the start to avoid repeating code. They are used by all
the sections below.


### Lipinski's rule (rule of five):
An empirical rule that predicts whether a chemical compound is likely to be an
orally active drug. It is called the "rule of five" because the key thresholds
are multiples of 5:

- Molecular weight ≤ 500 Da

- Log P (lipophilicity) ≤ 5

- H-bond donors (-OH, -NH groups) ≤ 5

- H-bond acceptors (N or O atoms) ≤ 10

If a compound violates more than one of these criteria, it is likely to have
poor oral absorption.

In [ ]:
def passes_lipinski(smiles):
    """True if the compound satisfies Lipinski's rule (drug-likeness)."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        return (Descriptors.MolWt(mol) < 500 and
                Descriptors.MolLogP(mol) < 5 and
                Lipinski.NumHDonors(mol) <= 5 and
                Lipinski.NumHAcceptors(mol) <= 10)
    except Exception:
        return False


def canonicalize_smiles(smi):
    """RDKit-canonicalized SMILES (key for deduplication). None if it fails."""
    try:
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol) if mol is not None else None
    except Exception:
        return None


def compute_features(smiles):
    """7 physicochemical descriptors + Morgan fingerprint (1024 bits)."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        desc = np.array([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Lipinski.NumHDonors(mol),
            Lipinski.NumHAcceptors(mol),
            rdMolDescriptors.CalcNumRotatableBonds(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
        ])
        gen = AllChem.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)
        fp = np.array(gen.GetFingerprintAsNumPy(mol))
        return np.concatenate([desc, fp])
    except Exception:
        return None


def get_morgan_fp(smiles, n_bits=FP_SIZE, radius=FP_RADIUS):
    """Morgan fingerprint as an RDKit bit-vector (for Tanimoto)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    gen = AllChem.GetMorganGenerator(radius=radius, fpSize=n_bits)
    return gen.GetFingerprint(mol)


def max_tanimoto_similarity(candidate_smiles, reference_fps):
    """Maximum Tanimoto of a candidate against the training set."""
    fp = get_morgan_fp(candidate_smiles)
    if fp is None:
        return None, 0
    sims = DataStructs.BulkTanimotoSimilarity(fp, reference_fps)
    return max(sims), sum(1 for s in sims if s >= 0.40)


def classify_confidence(sim):
    """Prediction confidence level based on the maximum Tanimoto."""
    if sim is None:
        return 'Error'
    elif sim >= 0.40:
        return 'High (inside domain)'
    elif sim >= 0.20:
        return 'Medium (marginal zone)'
    else:
        return 'Low (outside domain)'


print("Functions defined: Lipinski, canonicalization, features, Tanimoto, AD")

---
## 3. Dataset 1 — ChEMBL

We download the CFTR activities (CHEMBL4051) via the API: IC50, Ki and EC50.

In [ ]:
# Connect to the ChEMBL database
activities = new_client.activity

# Fields we are interested in: compound identifier, activity value,
# units, chemical structure and assay type.
FIELDS = ['molecule_chembl_id', 'standard_value', 'standard_units',
          'canonical_smiles', 'standard_type']

chembl_frames = []
# Biological measurements (IC50, Ki, EC50), one by one
for assay_type in ['IC50', 'Ki', 'EC50']:
    print(f"Downloading {assay_type}...")
    attempts, success = 0, False
    while attempts < 3 and not success:
        try:
            records = activities.filter(
                # Keep only CHEMBL4051, human (Homo sapiens) and
                # the current measurement (IC50, Ki or EC50).
                target_chembl_id='CHEMBL4051',
                standard_type=assay_type,
                target_organism='Homo sapiens'
            ).only(FIELDS)
            # Turn the downloaded records into a table
            df_assay = pd.DataFrame.from_records(records)
            # Tag these rows as coming from ChEMBL
            df_assay['source'] = 'ChEMBL'
            # Store the table in a list to concatenate them all later
            chembl_frames.append(df_assay)
            print(f"  -> {len(df_assay)} records")
            success = True
        except Exception as e:
            attempts += 1
            print(f"  Error (attempt {attempts}/3): {e}")
            if attempts < 3:
                time.sleep(10)

# Concatenate all tables (IC50, Ki and EC50) into one.
df_chembl_raw = pd.concat(chembl_frames, ignore_index=True)
# Save the raw table as a local cache
df_chembl_raw.to_csv(os.path.join(RESULTS_DIR, 'chembl_cftr_raw.csv'), index=False)
print(f"\nChEMBL raw: {len(df_chembl_raw)} records (copy saved)")

**ChEMBL cleaning** — nulls, nM units, conversion to pIC50, Lipinski, activity and deduplication.

In [ ]:
# Nulls and numeric conversion
df = df_chembl_raw.dropna(subset=['canonical_smiles', 'standard_value']).copy()
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])

# nM units only
df = df[df['standard_units'] == 'nM'].copy()

# Conversion to pIC50  (clip extreme values > 1e8 nM)
df['pIC50'] = -np.log10(df['standard_value'].clip(upper=1e8) * 1e-9)

# Lipinski + activity (pIC50 >= 5)
df['lipinski_ok'] = df['canonical_smiles'].apply(passes_lipinski)
df = df[df['lipinski_ok'] & (df['pIC50'] >= 5)].copy()

# Deduplicate: highest pIC50 per compound
df = df.sort_values('pIC50', ascending=False)
df_chembl = (df.drop_duplicates(subset='molecule_chembl_id', keep='first')
               [['canonical_smiles', 'pIC50']]
               .reset_index(drop=True))
df_chembl['source'] = 'ChEMBL'

print(f"ChEMBL clean: {len(df_chembl)} unique compounds")
print(f"pIC50: {df_chembl['pIC50'].min():.2f} - {df_chembl['pIC50'].max():.2f}"
      f"  (mean {df_chembl['pIC50'].mean():.2f})")

---
## 4. Dataset 2 — Papyrus

Papyrus is a high-quality bioactivity database. We download version 05.7 and
filter by CFTR (UniProt P13569).

**Note:** the first download is several GB and takes a while. After that it is
cached.

In [ ]:
from papyrus_scripts.download import download_papyrus
from papyrus_scripts.reader import read_papyrus
from papyrus_scripts.utils.IO import get_downloaded_versions

# Download only if not already cached
if '05.7' not in get_downloaded_versions():
    print("Downloading Papyrus 05.7 (several minutes)...")
    download_papyrus(version='05.7', only_pp=False)

# Read in chunks and filter CFTR
papyrus_data = read_papyrus(version='05.7', plusplus=False, chunksize=100000)

UNIPROT_CFTR = 'P13569'
chunks_filtered = [chunk[chunk['accession'] == UNIPROT_CFTR]
                   for chunk in papyrus_data]
chunks_filtered = [b for b in chunks_filtered if len(b) > 0]

df_papyrus_raw = pd.concat(chunks_filtered, ignore_index=True) if chunks_filtered else pd.DataFrame()
print(f"Papyrus CFTR raw: {len(df_papyrus_raw)} records")

**Papyrus cleaning** — high quality, column renaming, Lipinski, activity and
deduplication by median pIC50.

In [ ]:
# Keep only high-quality data (the 'Quality' column equals 'High')
df_p = df_papyrus_raw[df_papyrus_raw['Quality'] == 'High'].copy()
# Rename some columns so they match the other tables
df_p = df_p.rename(columns={
    'SMILES': 'canonical_smiles',
    'pchembl_value_Mean': 'pIC50',
    'connectivity': 'molecule_id',
})

# Apply the passes_lipinski function to each SMILES
df_p['lipinski_ok'] = df_p['canonical_smiles'].apply(passes_lipinski)
# Keep only compounds that pass Lipinski and have pIC50 >= 5
df_p = df_p[df_p['lipinski_ok'] & (df_p['pIC50'] >= 5)].copy()

df_papyrus = (df_p.groupby('molecule_id')
                  .agg({'canonical_smiles': 'first', 'pIC50': 'median'})
                  .reset_index()[['canonical_smiles', 'pIC50']])
# Tag these rows as coming from the Papyrus database.
df_papyrus['source'] = 'Papyrus'

print(f"Papyrus clean: {len(df_papyrus)} unique compounds")

---
## 5. Dataset 3 — BindingDB

BindingDB **has no API**: the TSV file is downloaded manually from its website
(search by target CFTR) and placed in the `data/` folder.

In [ ]:
# Path to the BindingDB TSV downloaded manually (see README)
BINDINGDB_PATH = os.path.join(DATA_DIR, 'bindingdb_cftr.tsv')

if os.path.exists(BINDINGDB_PATH):
    df_bdb_raw = pd.read_csv(BINDINGDB_PATH, sep='\t',
                             on_bad_lines='skip', low_memory=False)
    print(f"BindingDB raw: {len(df_bdb_raw)} records")
    USE_BINDINGDB = True
else:
    print(f"[Warning] '{BINDINGDB_PATH}' not found.")
    print("The notebook will continue with ChEMBL + Papyrus only.")
    df_bdb_raw = pd.DataFrame()
    USE_BINDINGDB = False

**BindingDB cleaning** — pIC50 computation (priority IC50 > Ki > Kd > EC50),
Lipinski, activity and deduplication.

In [ ]:
# Convert to a float; if it fails (not a number), return NaN
def clean_nM_value(v):
    try:
        return float(str(v).replace('<', '').replace('>', '').strip())
    except Exception:
        return np.nan

# Scan several columns, take the first positive numeric value and convert nM to pIC50
def pIC50_from_bindingdb(row):
    for col in ['IC50 (nM)', 'Ki (nM)', 'Kd (nM)', 'EC50 (nM)']:
        v = clean_nM_value(row.get(col, np.nan))
        if not np.isnan(v) and v > 0:
            return -np.log10(v * 1e-9)
    return np.nan


if USE_BINDINGDB:
    # Add a pIC50 column computed with the function above for each row.
    df_bdb_raw['pIC50'] = df_bdb_raw.apply(pIC50_from_bindingdb, axis=1)
    # Drop rows without SMILES (chemical structure) or pIC50
    df_b = df_bdb_raw.dropna(subset=['Ligand SMILES', 'pIC50']).copy()
    # Rename that column so it matches the other tables.
    df_b = df_b.rename(columns={'Ligand SMILES': 'canonical_smiles'})
    # Keep only potent compounds (pIC50 >= 5).
    df_b = df_b[df_b['pIC50'] >= 5].copy()

    # Apply Lipinski's rule to each SMILES
    df_b['lipinski_ok'] = df_b['canonical_smiles'].apply(passes_lipinski)
    # Keep only those that pass Lipinski.
    df_b = df_b[df_b['lipinski_ok']].copy()

    # Canonicalize each SMILES to a unique, ordered form
    df_b['canonical_key'] = df_b['canonical_smiles'].apply(canonicalize_smiles)
    # Drop those that could not be canonicalized
    df_b = df_b.dropna(subset=['canonical_key'])

    df_bindingdb = (df_b.groupby('canonical_key')
                        .agg({'canonical_smiles': 'first', 'pIC50': 'median'})
                        .reset_index()[['canonical_smiles', 'pIC50']])
    df_bindingdb['source'] = 'BindingDB'
    print(f"BindingDB clean: {len(df_bindingdb)} unique compounds")
else:
    # Create an empty table with the same columns and warn it was skipped
    df_bindingdb = pd.DataFrame(columns=['canonical_smiles', 'pIC50', 'source'])
    print("BindingDB skipped.")

---
## 6. Merging the 3 databases

We join the three sources. We generate a canonical key (canonicalized SMILES)
and, if a compound appears in several databases, we keep the **median** of its
pIC50. This is the dataset used to train the model.

In [ ]:
# Join the three sources
df_all = pd.concat([df_chembl, df_papyrus, df_bindingdb], ignore_index=True)

# Canonical key for deduplication
df_all['canonical_key'] = df_all['canonical_smiles'].apply(canonicalize_smiles)
df_all = df_all.dropna(subset=['canonical_key'])

# Deduplicate: median pIC50 + list of sources
df_dataset = (df_all.groupby('canonical_key')
                    .agg({'canonical_smiles': 'first',
                          'pIC50': 'median',
                          'source': lambda x: ','.join(sorted(set(x)))})
                    .reset_index())

df_dataset.to_csv(os.path.join(RESULTS_DIR, 'dataset_full.csv'), index=False)

print(f"FULL DATASET: {len(df_dataset)} unique compounds")
print(f"pIC50: {df_dataset['pIC50'].min():.2f} - {df_dataset['pIC50'].max():.2f}"
      f"  (mean {df_dataset['pIC50'].mean():.2f})")
print("\nCompounds by source combination:")
print(df_dataset['source'].value_counts())
print("\nSaved: results/dataset_full.csv")

---
## 7. Model 1 — ChEMBL-only baseline

Before training the model on the three merged databases, we train a
**ChEMBL-only baseline model** so we can assess the impact of adding Papyrus and
BindingDB to the dataset.

**Goal of the comparison:** check whether merging the three sources improves the
predictive ability of the model relative to using ChEMBL alone.

**Methodological caveat:** the two models are trained on datasets of different
size and composition, so their R² and RMSE are not strictly comparable in
absolute terms. The comparison is indicative.

In [ ]:
# Model 1 — ChEMBL-only baseline
print("Computing features for the ChEMBL dataset...")
X_list_c = df_chembl['canonical_smiles'].apply(compute_features)
valid_c = X_list_c.notna()
X_chembl = np.array(X_list_c[valid_c].tolist())
y_chembl = df_chembl['pIC50'][valid_c].values
print(f"Features computed: {X_chembl.shape[0]} compounds, {X_chembl.shape[1]} features")

# Same hyperparameters as the merged model so the comparison is fair
rf_chembl = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_leaf=2,
    random_state=SEED, n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Cross-validation (5-fold)
r2_cv_c = cross_val_score(rf_chembl, X_chembl, y_chembl, cv=cv, scoring='r2')
rmse_cv_c = np.sqrt(-cross_val_score(rf_chembl, X_chembl, y_chembl, cv=cv,
                                     scoring='neg_mean_squared_error'))
print(f"\n[ChEMBL only] R2 CV:   {r2_cv_c.mean():.3f} +/- {r2_cv_c.std():.3f}")
print(f"[ChEMBL only] RMSE CV: {rmse_cv_c.mean():.3f} +/- {rmse_cv_c.std():.3f}")

# Hold-out 20%
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_chembl, y_chembl, test_size=0.20, random_state=SEED
)
rf_chembl.fit(X_tr_c, y_tr_c)
y_pred_c = rf_chembl.predict(X_te_c)
r2_ho_c = r2_score(y_te_c, y_pred_c)
rmse_ho_c = np.sqrt(mean_squared_error(y_te_c, y_pred_c))
print(f"[ChEMBL only] R2 hold-out:   {r2_ho_c:.3f}")
print(f"[ChEMBL only] RMSE hold-out: {rmse_ho_c:.3f}")

# Store the metrics for the comparison table
metrics_chembl = {
    'n_compounds': len(y_chembl),
    'r2_cv': r2_cv_c.mean(), 'r2_cv_std': r2_cv_c.std(),
    'rmse_cv': rmse_cv_c.mean(), 'rmse_cv_std': rmse_cv_c.std(),
    'r2_ho': r2_ho_c, 'rmse_ho': rmse_ho_c,
}

---
## 8. Model 2 — QSAR with the merged dataset (ChEMBL + Papyrus + BindingDB)

We compute the features of the merged dataset and train a Random Forest with the
same hyperparameters as Model 1, so the comparison is fair.

For evaluation we use **two complementary methods**:
- **Cross-validation (5-fold)**: estimates the average performance over the whole dataset.
- **Hold-out (20%)**: a fixed split the model does not see during training, to
  obtain an independent value.

After evaluation, the final model is retrained on 100% of the data —
**this merged Model 2 is the one used to predict the candidates**.

In [ ]:
print("Computing features for the full dataset...")
# For each SMILES (chemical structure) in df_dataset, apply compute_features.
X_list = df_dataset['canonical_smiles'].apply(compute_features)
# Mark which compounds produced valid features
valid = X_list.notna()
# Keep only the valid ones
X = np.array(X_list[valid].tolist())
# Take the pIC50 column (potency of each compound) for valid rows only, as an array
y = df_dataset['pIC50'][valid].values
print(f"Valid compounds: {len(y)}  |  X dimensions: {X.shape}")

**Hyperparameters.** We keep `n_estimators=200` and `min_samples_leaf=2`
(200 trees give stability without excessive cost, and requiring 2 samples per
leaf reduces overfitting). A larger number of trees barely changes the result.

---

## 5-fold cross-validation over the whole dataset → R² ≈ 0.649.

In [ ]:
# Model and validation scheme
rf = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_leaf=2,
    random_state=SEED, n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Cross-validation
r2_cv = cross_val_score(rf, X, y, cv=cv, scoring='r2')
rmse_cv = np.sqrt(-cross_val_score(rf, X, y, cv=cv,
                                   scoring='neg_mean_squared_error'))
print("Cross-validation (k=5):")
print(f"  R2   = {r2_cv.mean():.3f} +/- {r2_cv.std():.3f}")
print(f"  RMSE = {rmse_cv.mean():.3f} +/- {rmse_cv.std():.3f}")

---

## Hold-out: train on 80%, measure on the unseen 20% → R² ≈ 0.686.

In [ ]:
# Hold-out test (20% the model does not see during training)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)

print("Hold-out test (20%):")
print(f"  R2   = {r2_score(y_te, y_pred):.3f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(y_te, y_pred)):.3f}")
print("\nIf the hold-out R2 is close to the CV R2, there is no severe overfitting.")

In [ ]:
# Store the merged-model metrics for the comparison
metrics_fused = {
    'n_compounds': len(y),
    'r2_cv': r2_cv.mean(), 'r2_cv_std': r2_cv.std(),
    'rmse_cv': rmse_cv.mean(), 'rmse_cv_std': rmse_cv.std(),
    'r2_ho': r2_score(y_te, y_pred), 'rmse_ho': np.sqrt(mean_squared_error(y_te, y_pred)),
}

**Final model.** We retrain it on 100% of the data. This is the model used to
predict the candidates.

In [ ]:
rf.fit(X, y)
print("Final model trained on", len(y), "compounds from the full dataset.")

# Fingerprints of the full dataset (for the applicability domain)
print("Computing fingerprints for the applicability domain...")
fps_train = [fp for fp in
             (get_morgan_fp(s) for s in df_dataset['canonical_smiles'])
             if fp is not None]
print(f"Reference fingerprints: {len(fps_train)}")

---
### Comparison: Model 1 (ChEMBL) vs Model 2 (Merged)

Summary of both models' performance.

In [ ]:
# Comparison table
comparison_table = pd.DataFrame({
    'Model': ['Model 1 — ChEMBL only', 'Model 2 — ChEMBL+Papyrus+BindingDB'],
    'N compounds': [metrics_chembl['n_compounds'], metrics_fused['n_compounds']],
    'R2 (CV 5-fold)': [f"{metrics_chembl['r2_cv']:.3f} +/- {metrics_chembl['r2_cv_std']:.3f}",
                       f"{metrics_fused['r2_cv']:.3f} +/- {metrics_fused['r2_cv_std']:.3f}"],
    'RMSE (CV 5-fold)': [f"{metrics_chembl['rmse_cv']:.3f} +/- {metrics_chembl['rmse_cv_std']:.3f}",
                         f"{metrics_fused['rmse_cv']:.3f} +/- {metrics_fused['rmse_cv_std']:.3f}"],
    'R2 (hold-out 20%)': [f"{metrics_chembl['r2_ho']:.3f}",
                          f"{metrics_fused['r2_ho']:.3f}"],
    'RMSE (hold-out 20%)': [f"{metrics_chembl['rmse_ho']:.3f}",
                            f"{metrics_fused['rmse_ho']:.3f}"],
})
print(comparison_table.to_string(index=False))
comparison_table.to_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'), index=False)

---
## 9. Candidates and validation controls

We define four groups of molecules:

- **7 candidates** — identified from the PPI network analysis (STRING) and
  DrugBank. These are the ones evaluated and ranked.
- **Positive controls** — Ivacaftor and the approved CFTR correctors
  (Lumacaftor, Tezacaftor, Elexacaftor). They act as a **reference ceiling**: the
  model should give them a high pIC50.
- **Negative controls** — drugs with no known CFTR interaction
  (Paracetamol, Aspirin, Metformin). They act as a **baseline**: the model
  should give them a low pIC50.
- **R406** — active metabolite of Fostamatinib (which is a prodrug).

Important: **the controls do NOT enter the ranking**. They are used only to
validate the model and the docking. The ranking in section 13 contains only the
7 candidates.

In [ ]:
# --- 7 candidates + Ivacaftor ---
candidates = [
    # (name, hub_target, SMILES, DrugBank ID, indication, group)
    ('Bumetanide',       'CFTR',     'CCCCNC1=C(OC2=CC=CC=C2)C(=CC(=C1)C(O)=O)S(N)(=O)=O',
                                     'DB00887', 'Diuretic',        'Candidate'),
    ('Glyburide',        'CFTR',     'COC1=C(C=C(Cl)C=C1)C(=O)NCCC1=CC=C(C=C1)S(=O)(=O)NC(=O)NC1CCCCC1',
                                     'DB01016', 'Antidiabetic',    'Candidate'),
    ('Ibuprofen',        'CFTR',     'CC(C)CC1=CC=C(C=C1)C(C)C(O)=O',
                                     'DB01050', 'NSAID',           'Candidate'),
    ('Cromoglicic_acid', 'HSP90AA1', 'OC(COC1=CC=CC2=C1C(=O)C=C(O2)C(O)=O)COC1=CC=CC2=C1C(=O)C=C(O2)C(O)=O',
                                     'DB01003', 'Antiallergic',    'Candidate'),
    ('Fostamatinib',     'PRKACA',   'COC1=CC(NC2=NC=C(F)C(NC3=NC4=C(OC(C)(C)C(=O)N4COP(O)(O)=O)C=C3)=N2)=CC(OC)=C1OC',
                                     'DB12010', 'Immunomodulator', 'Candidate'),
    ('Dasatinib',        'HSPA8',    'CC1=NC(NC2=NC=C(S2)C(=O)NC2=C(C)C=CC=C2Cl)=CC(=N1)N1CCN(CCO)CC1',
                                     'DB01254', 'Antineoplastic',  'Candidate'),
    ('Artenimol',        'HSPA8',    '[H][C@@]12CC[C@@H](C)[C@]3([H])CC[C@@]4(C)OO[C@@]13[C@]([H])(O[C@H](O)[C@@H]2C)O4',
                                     'DB11638', 'Antimalarial',    'Candidate'),
    ('Ivacaftor',        'CFTR',     'CC(C)(C)C1=CC(=C(O)C=C1NC(=O)C1=CNC2=CC=CC=C2C1=O)C(C)(C)C',
                                     'DB08820', 'CF - potentiator', 'Control (+)'),
]

# --- Additional positive controls: approved correctors ---
positive_controls = [
    ('Lumacaftor',  'CFTR', 'CC1=CC=C(NC(=O)C2(CC2)C2=CC=C3OC(F)(F)OC3=C2)N=C1C1=CC(=CC=C1)C(O)=O',
                            'DB09280', 'CF - corrector', 'Control (+)'),
    ('Tezacaftor',  'CFTR', 'CC(C)(CO)C1=CC2=CC(NC(=O)C3(CC3)C3=CC=C4OC(F)(F)OC4=C3)=C(F)C=C2N1C[C@@H](O)CO',
                            'DB11712', 'CF - corrector', 'Control (+)'),
    ('Elexacaftor', 'CFTR', 'C[C@@H]1CN(C2=NC(=CC=C2C(=O)NS(=O)(=O)C2=CN(C)N=C2C)N2C=CC(OCC(C)(C)C(F)(F)F)=N2)C(C)(C)C1',
                            'DB15444', 'CF - corrector', 'Control (+)'),
]

# --- Negative controls: no known CFTR interaction ---
negative_controls = [
    ('Paracetamol', 'None', 'CC(=O)NC1=CC=C(O)C=C1',
                            'DB00316', 'Negative control', 'Control (-)'),
    ('Aspirin',     'None', 'CC(=O)OC1=CC=CC=C1C(O)=O',
                            'DB00945', 'Negative control', 'Control (-)'),
    ('Metformin',   'None', 'CN(C)C(=N)NC(N)=N',
                            'DB00331', 'Negative control', 'Control (-)'),
]

# --- R406: active metabolite of Fostamatinib ---
r406 = [
    ('R406', 'PRKACA', 'COc1cc(Nc2ncc(F)c(Nc3ccc4c(n3)NC(=O)C(C)(C)O4)n2)cc(OC)c1OC',
                       'N/A', 'Active metabolite', 'Metabolite'),
]

cols = ['name', 'hub_target', 'smiles', 'drugbank_id', 'indication', 'group']

# Single table with all molecules (candidates + controls)
df_mol = pd.DataFrame(candidates + positive_controls + negative_controls + r406,
                      columns=cols)

print("Molecules defined:")
print(df_mol['group'].value_counts())
df_mol[['name', 'group', 'hub_target', 'indication']]

---
## 10. Applicability domain (Tanimoto)

We compute the maximum Tanimoto similarity of **each molecule** (candidates and
controls) against the **full dataset**. We reuse the `fps_train` fingerprints
computed in section 8. A high Tanimoto means the model has seen similar
molecules and its prediction is more reliable.

In [ ]:
ad_results = df_mol['smiles'].apply(lambda s: max_tanimoto_similarity(s, fps_train))
df_mol['tanimoto_max'] = [round(t[0], 3) if t[0] else None for t in ad_results]
df_mol['prediction_confidence'] = df_mol['tanimoto_max'].apply(classify_confidence)

print("Applicability domain (against the full dataset):\n")
print(df_mol[['name', 'group', 'tanimoto_max', 'prediction_confidence']]
      .to_string(index=False))

---
## 11. pIC50 prediction and integration with docking

We predict the pIC50 of **all molecules** with the full model (section 8) and add
the docking energies computed with AutoDock Vina.

> Note: the docking energies below are **hard-coded results** obtained outside
> this notebook (AutoDock Vina on PDB 6MSM, apo WT CFTR, and on an
> AlphaFold2 ΔF508 model). See the README for details.

In [ ]:
# pIC50 prediction with the full model (all molecules)
feats = df_mol['smiles'].apply(compute_features)
X_mol = np.array(feats.tolist())
df_mol['pIC50_pred'] = rf.predict(X_mol)

# Docking energies (AutoDock Vina) — computed externally, see README
docking_wt = {
    'Bumetanide': -7.210, 'Glyburide': -10.105, 'Ibuprofen': -6.593,
    'Cromoglicic_acid': -9.347, 'Fostamatinib': -9.009, 'Dasatinib': -9.131,
    'Artenimol': -6.613, 'Ivacaftor': -9.493,
    'Lumacaftor': -10.775, 'Tezacaftor': -10.213, 'Elexacaftor': -9.493,
    'Paracetamol': -6.164, 'Aspirin': -5.701, 'Metformin': -5.268,
    'R406': -9.177,
}
docking_delta = {
    'Bumetanide': -7.147, 'Glyburide': -9.684, 'Ibuprofen': -6.267,
    'Cromoglicic_acid': -9.326, 'Fostamatinib': -8.781, 'Dasatinib': -9.242,
    'Artenimol': -6.886, 'Ivacaftor': -9.651,
    'Lumacaftor': -10.293, 'Tezacaftor': -10.168, 'Elexacaftor': -9.379,
    'Paracetamol': -5.608, 'Aspirin': -5.432, 'Metformin': -5.798,
    'R406': -9.301,
}
df_mol['dock_wt'] = df_mol['name'].map(docking_wt)
df_mol['dock_delta'] = df_mol['name'].map(docking_delta)
df_mol['dock_mean'] = (df_mol['dock_wt'] + df_mol['dock_delta']) / 2

print(df_mol[['name', 'group', 'pIC50_pred', 'tanimoto_max',
              'dock_mean']].round(3).to_string(index=False))

---
## 12. Model validation with the controls

Before trusting the ranking, we check that the model behaves consistently with
the controls:

- The **positive controls** (approved modulators/correctors) should receive a
  **high** predicted pIC50 and a **low** docking energy (good binding).
- The **negative controls** should receive a **low** pIC50 and a **high** docking
  energy (weak binding) — they mark the baseline of nonspecific energy.
- **R406** should score at least as well as Fostamatinib (its prodrug).

If this holds, the model separates actives from inactives well and the candidate
ranking is more credible.

In [ ]:
# Summary by group
summary = (df_mol.groupby('group')
           .agg(n=('name', 'count'),
                mean_pIC50=('pIC50_pred', 'mean'),
                mean_dock=('dock_mean', 'mean'))
           .round(3))
print("Summary by group:\n")
print(summary.to_string())

baseline_neg = df_mol.loc[df_mol['group'] == 'Control (-)', 'dock_mean'].mean()
print(f"\nDocking baseline (negative controls): {baseline_neg:.3f} kcal/mol")

# Automatic checks
pos_ctrl = df_mol[df_mol['group'] == 'Control (+)']
neg_ctrl = df_mol[df_mol['group'] == 'Control (-)']
ok_pic50 = pos_ctrl['pIC50_pred'].mean() > neg_ctrl['pIC50_pred'].mean()
ok_dock = pos_ctrl['dock_mean'].mean() < neg_ctrl['dock_mean'].mean()
print("\nChecks:")
print(f"  pIC50 control(+) > control(-) : {'OK' if ok_pic50 else 'REVIEW'}")
print(f"  docking control(+) better than control(-) : {'OK' if ok_dock else 'REVIEW'}")

# R406 vs Fostamatinib
r406_d = df_mol.loc[df_mol['name'] == 'R406', 'dock_mean'].iloc[0]
fosta_d = df_mol.loc[df_mol['name'] == 'Fostamatinib', 'dock_mean'].iloc[0]
print(f"\nR406 docking = {r406_d:.3f}  vs  Fostamatinib = {fosta_d:.3f} kcal/mol")

**Validation plot** — predicted pIC50 vs docking, colored by group. Positive
controls should fall top-left (best) and negative controls bottom-right (worst).

In [ ]:
# Create a figure
fig, ax = plt.subplots(figsize=(8, 6))


# Candidate    -> blue, circle
# Control (+)  -> green, square
# Control (-)  -> red, inverted triangle
# Metabolite   -> purple, diamond
style_map = {
    'Candidate':   ('#3498db', 'o'),
    'Control (+)': ('#2ecc71', 's'),
    'Control (-)': ('#e74c3c', 'v'),
    'Metabolite':  ('#9b59b6', 'D'),
}

# Loop over each group
for group, (color, marker) in style_map.items():
    # take only the rows of that group
    sub = df_mol[df_mol['group'] == group]
    # plot the points using the matching color and marker
    ax.scatter(sub['dock_mean'], sub['pIC50_pred'],
               color=color, marker=marker, s=90, edgecolor='black',
               label=group, zorder=3)
    # loop over each compound in the current group and add a label with its name
    for _, r in sub.iterrows():
        ax.annotate(r['name'], (r['dock_mean'], r['pIC50_pred']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7)

# Axis labels and title
ax.set_xlabel('Mean docking (kcal/mol)  ->  better to the left')
ax.set_ylabel('Predicted pIC50 (full model)')
ax.set_title('Model validation with controls\n'
             '(positives = top-left;  negatives = bottom-right)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
# Save and show
plt.savefig(os.path.join(RESULTS_DIR, 'fig_control_validation.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved: results/fig_control_validation.png")

---
## 13. Final ranking

The ranking contains **only the 7 candidates**. Ivacaftor is used as a reference
line and the other controls do not enter (section 11).

**Integrated score formula:**
$$\text{score} = 0.60 \times \text{dock\_norm} + 0.40 \times \text{pIC50\_norm}$$

We normalize both metrics to the range [0, 1] with MinMaxScaler. For docking we
invert the sign: a more negative value (better binding) gives a higher score.

**Important note on normalization.** The score is **relative to the candidate
group**: MinMaxScaler assigns 0 to the worst and 1 to the best of the 7. So a
score of 0.000 means "the worst of this group", not "zero affinity". If
candidates were added or removed, the scores would change.

In [ ]:
# Take from df_mol the 7 candidates + Ivacaftor (they already have pIC50, AD and docking)
candidate_names = ['Bumetanide', 'Glyburide', 'Ibuprofen', 'Cromoglicic_acid',
                   'Fostamatinib', 'Dasatinib', 'Artenimol', 'Ivacaftor']
df_cand = df_mol[df_mol['name'].isin(candidate_names)].copy().reset_index(drop=True)

# Normalization (docking inverted: more negative = better)
scaler_dock = MinMaxScaler()
scaler_pic50 = MinMaxScaler()
df_cand['dock_norm'] = 1 - scaler_dock.fit_transform(df_cand[['dock_mean']])
df_cand['pic50_norm'] = scaler_pic50.fit_transform(df_cand[['pIC50_pred']])

# Integrated score 60/40
df_cand['score_final'] = 0.60 * df_cand['dock_norm'] + 0.40 * df_cand['pic50_norm']

# Ranking (Ivacaftor excluded: it is a control, not a candidate)
ivacaftor = df_cand[df_cand['name'] == 'Ivacaftor'].iloc[0]
df_rank = df_cand[df_cand['name'] != 'Ivacaftor'].copy()
df_rank = df_rank.sort_values('score_final', ascending=False).reset_index(drop=True)
df_rank['rank'] = df_rank.index + 1

print("=" * 80)
print("FINAL RANKING — model trained on ChEMBL + Papyrus + BindingDB")
print("=" * 80)
cols = ['rank', 'name', 'dock_mean', 'pIC50_pred',
        'tanimoto_max', 'prediction_confidence', 'score_final']
print(df_rank[cols].round(3).to_string(index=False))

print(f"\nIvacaftor reference (control): score = {ivacaftor['score_final']:.3f}")

**Weight sensitivity analysis.** We recompute the score with different
combinations to check that the ranking is robust.

In [ ]:
print("Score sensitivity to the weights:\n")
for w_dock, w_qsar in [(0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (1.0, 0.0)]:
    df_rank[f's_{int(w_dock*100)}'] = (w_dock * df_rank['dock_norm']
                                       + w_qsar * df_rank['pic50_norm'])

print(df_rank[['name', 's_50', 's_60', 's_70', 's_100']]
      .round(3).to_string(index=False))
print("\nIf the top-3 stays the same across the four columns, the ranking is robust.")

---
## 14. Visualizations

### 14.1 Final ranking colored by applicability domain

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

color_map = {'High (inside domain)':   '#2ecc71',
             'Medium (marginal zone)': '#f39c12',
             'Low (outside domain)':   '#e74c3c'}

# Reverse order so that #1 ends up on top
names  = df_rank['name'].tolist()[::-1]
scores = df_rank['score_final'].tolist()[::-1]
docks  = df_rank['dock_mean'].tolist()[::-1]
conf   = df_rank['prediction_confidence'].tolist()[::-1]
colors = [color_map.get(c, 'steelblue') for c in conf]

bars = ax.barh(names, scores, color=colors, edgecolor='black', height=0.6)

ax.axvline(ivacaftor['score_final'], color='navy', linestyle='--', linewidth=1.5,
           label=f"Ivacaftor (ref.) = {ivacaftor['score_final']:.3f}")

# Label: normalized score + real docking energy
for bar, s, d in zip(bars, scores, docks):
    ax.text(s + 0.01, bar.get_y() + bar.get_height()/2,
            f'{s:.3f}  ({d:.1f} kcal/mol)', va='center', fontsize=8)

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='AD: Inside domain'),
    mpatches.Patch(color='#f39c12', label='AD: Marginal zone'),
    mpatches.Patch(color='#e74c3c', label='AD: Outside domain'),
]
ax.legend(handles=legend_patches + [ax.lines[0]], loc='lower right', fontsize=8)
ax.set_xlabel('Integrated score (0.60 x docking + 0.40 x pIC50)')
ax.set_title('Final ranking of CF repositioning candidates\n'
             '(model: ChEMBL + Papyrus + BindingDB)')
ax.set_xlim(0, 1.25)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_final_ranking.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved: results/fig_final_ranking.png")

### 14.2 Descriptor importance in the model

The first 7 values correspond to the physicochemical descriptors; the rest are
Morgan fingerprint bits.

In [ ]:
desc_names = ['MolWt', 'LogP', 'TPSA', 'HDonors',
              'HAcceptors', 'RotBonds', 'AromRings']
feat_names = desc_names + [f'FP_{i}' for i in range(FP_SIZE)]

importances = pd.Series(rf.feature_importances_, index=feat_names)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
top15[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Importance')
ax.set_title('Top 15 most influential descriptors (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved: results/fig_feature_importance.png")

---
## 15. Export results

We save the final ranking and the control-validation table as CSV to include
them in the thesis.

In [ ]:
# Candidate ranking (+ Ivacaftor as reference)
cols_export = ['name', 'hub_target', 'drugbank_id', 'indication',
               'dock_wt', 'dock_delta', 'dock_mean',
               'pIC50_pred', 'tanimoto_max', 'prediction_confidence',
               'dock_norm', 'pic50_norm', 'score_final']

df_export = pd.concat([
    df_rank[cols_export],
    df_cand[df_cand['name'] == 'Ivacaftor'][cols_export]
], ignore_index=True)
df_export.to_csv(os.path.join(RESULTS_DIR, 'final_ranking.csv'), index=False)
print("Exported: results/final_ranking.csv")
print(df_export.round(3).to_string(index=False))

# Validation table with all groups
cols_val = ['name', 'group', 'hub_target', 'pIC50_pred',
            'tanimoto_max', 'prediction_confidence', 'dock_mean']
df_mol[cols_val].round(3).to_csv(os.path.join(RESULTS_DIR, 'control_validation.csv'), index=False)
print("\nExported: results/control_validation.csv")

---

# 16. Comparative figure of the four PLIP profiles

A grouped bar chart: x-axis = the 4 top candidates, y-axis = number of
interactions, one bar per type (H-bonds, hydrophobic, pi-stacking, salt bridges).

> Note: the interaction counts below are **hard-coded** results from PLIP run on
> the docked poses (CFTR 6MSM), not computed in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plip_candidates = ['Glyburide', 'Cromoglicic acid', 'Fostamatinib', 'Dasatinib']
h_bonds = [3, 6, 11, 1]
hydrophobic = [6, 3, 2, 3]
pi_stacking = [2, 3, 1, 1]
salt_bridges = [0, 1, 1, 0]

x = np.arange(len(plip_candidates))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - 1.5*width, h_bonds, width, label='H-bonds', color='#2E86AB')
ax.bar(x - 0.5*width, hydrophobic, width, label='Hydrophobic', color='#E07A5F')
ax.bar(x + 0.5*width, pi_stacking, width, label='pi-stacking', color='#81B29A')
ax.bar(x + 1.5*width, salt_bridges, width, label='Salt bridges', color='#F2CC8F')

ax.set_xlabel('Candidate')
ax.set_ylabel('Number of interactions')
ax.set_title('Molecular interaction profile per candidate (PLIP on CFTR-6MSM)')
ax.set_xticks(x)
ax.set_xticklabels(plip_candidates)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_plip_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()